In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import read_csv
# Download latest version
path = kagglehub.dataset_download("hernan4444/anime-recommendation-database-2020")

print("Path to dataset files:", path)

100%|██████████| 661M/661M [00:09<00:00, 76.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hernan4444/anime-recommendation-database-2020/versions/7


In [ ]:
import os

# Load Kaggle datasets
user_completed_animelist = read_csv(os.path.join(path, "rating_complete.csv"))

In [ ]:
user_completed_animelist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57633278 entries, 0 to 57633277
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 1.3 GB


This is our biggest dataset and for the sake of processing power and time, I would be better to take a sample to use for our analysis. To do this we would need to make sure that we maintain all the data for each user we include in the sample.

Right now the dataset is organised by anime, meaning that each row is an anime entry made by a user. Before we sample we should find a way to group together all the entries made by each user.

In [ ]:
user_animelist_ids = pd.DataFrame({"user_id":user_animelist['user_id'].unique()})
user_animelist_ids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325770 entries, 0 to 325769
Data columns (total 1 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   user_id  325770 non-null  int64
dtypes: int64(1)
memory usage: 2.5 MB


In [ ]:
user_animelist_ids = user_animelist_ids.sample(frac=0.5)
user_animelist_ids.head(10)

,user_id
288942,313452
262504,284673
182537,198103
314272,340881
191279,207561
6311,6873
122294,132752
75803,82258
55298,60004
119709,129971


In [ ]:
# my attempt at sampling, looping was a bad idea, continued to run for almost half an hour with no clear end (at least it seemed to work though)
user_entries = pd.DataFrame(columns=["user_id","anime_entries"])
for idx, row in user_animelist_ids.iterrows():
    id = row['user_id']
    anime_entries=[]
    anime_data = {}
    for idx2, row2 in user_animelist.iterrows():
        if id == row2['user_id']:
          anime_data = {}
          anime_data['id']=row2['anime_id']
          anime_data['rating']=row2['rating']
          anime_data['status']=row2['watching_status']
          anime_data['episodes']=row2['watched_episodes']
          anime_entries.append(anime_data)
    pd.concat([user_entries, pd.DataFrame.from_records([{ "user_id": row['user_id'], "anime_entries": anime_entries }])])

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# SAMPLING USER ANIMELIST DATASET
# ============================================================================

"""
SAMPLING STRATEGY:
------------------
Goal: Reduce dataset size while preserving complete user behavior

The challenge: Each user has multiple rows (one per anime they watched)
Solution: Sample complete users, not individual rows

This ensures:
1. All anime for each sampled user are included
2. User behavior is preserved completely
3. No orphaned data (anime with missing user context)
"""

def get_dataset_info(user_animelist_df):
    """
    [STEP 1] Analyze the current dataset

    Show:
    - Total rows
    - Unique users
    - Anime per user statistics
    - Memory usage
    """
    print("\n" + "="*70)
    print("USER ANIMELIST DATASET SAMPLING")
    print("="*70)

    print("\n[STEP 1] Current Dataset Information")
    print("-" * 70)

    total_rows = len(user_animelist_df)
    unique_users = user_animelist_df['user_id'].nunique()
    unique_anime = user_animelist_df['anime_id'].nunique()

    print(f"\nTotal rows: {total_rows:,}")
    print(f"Unique users: {unique_users:,}")
    print(f"Unique anime: {unique_anime:,}")
    print(f"Memory usage: {user_animelist_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    # Anime per user statistics
    anime_per_user = user_animelist_df.groupby('user_id').size()

    print(f"\nAnime per User Statistics:")
    print(f"  Mean: {anime_per_user.mean():.1f}")
    print(f"  Median: {anime_per_user.median():.1f}")
    print(f"  Min: {anime_per_user.min()}")
    print(f"  Max: {anime_per_user.max()}")
    print(f"  Std Dev: {anime_per_user.std():.1f}")

    return {
        'total_rows': total_rows,
        'unique_users': unique_users,
        'unique_anime': unique_anime,
        'anime_per_user': anime_per_user
    }


def sample_users(user_animelist_df, num_users=10000, random_state=42):
    """
    [STEP 2] Sample complete users from the dataset

    This approach:
    1. Gets list of all unique users
    2. Randomly samples N users
    3. Filters dataframe to only include those users
    4. Preserves all anime for each sampled user

    Args:
    - user_animelist_df: The full user animelist dataset
    - num_users: Number of users to sample
    - random_state: For reproducibility

    Returns:
    - sampled_df: Dataset containing only sampled users (all their anime)
    """
    print("\n[STEP 2] Sampling Complete Users")
    print("-" * 70)

    # Get all unique users
    all_users = user_animelist_df['user_id'].unique()
    print(f"\nTotal unique users available: {len(all_users):,}")

    # Sample users
    np.random.seed(random_state)
    sampled_user_ids = np.random.choice(all_users, size=num_users, replace=False)
    print(f"Sampled users: {len(sampled_user_ids):,}")

    # Filter dataframe to only include sampled users
    print(f"\nFiltering dataset to sampled users...")
    sampled_df = user_animelist_df[user_animelist_df['user_id'].isin(sampled_user_ids)].copy()

    print(f"✓ Sampling complete!")

    return sampled_df, sampled_user_ids


def analyze_sampled_dataset(sampled_df, original_info):
    """
    [STEP 3] Analyze the sampled dataset

    Compare sampled dataset with original:
    - Size reduction
    - User and anime counts
    - Memory usage
    - Data distribution preservation
    """
    print("\n[STEP 3] Sampled Dataset Analysis")
    print("-" * 70)

    sampled_rows = len(sampled_df)
    sampled_users = sampled_df['user_id'].nunique()
    sampled_anime = sampled_df['anime_id'].nunique()
    sampled_memory = sampled_df.memory_usage(deep=True).sum() / 1e9

    # Calculate reduction percentages
    row_reduction = (1 - sampled_rows / original_info['total_rows']) * 100
    user_reduction = (1 - sampled_users / original_info['unique_users']) * 100
    anime_reduction = (1 - sampled_anime / original_info['unique_anime']) * 100
    memory_reduction = row_reduction  # Same as row reduction

    print(f"\nSize Comparison:")
    print(f"  Original rows: {original_info['total_rows']:,}")
    print(f"  Sampled rows: {sampled_rows:,}")
    print(f"  Reduction: {row_reduction:.1f}%")

    print(f"\nUsers:")
    print(f"  Original: {original_info['unique_users']:,}")
    print(f"  Sampled: {sampled_users:,}")
    print(f"  Reduction: {user_reduction:.1f}%")

    print(f"\nAnime Included:")
    print(f"  Original: {original_info['unique_anime']:,}")
    print(f"  Sampled: {sampled_anime:,}")
    print(f"  Reduction: {anime_reduction:.1f}%")

    print(f"\nMemory Usage:")
    print(f"  Original: {original_info['total_rows'] * 40 / 1e9:.2f} GB (estimated)")
    print(f"  Sampled: {sampled_memory:.2f} GB")
    print(f"  Reduction: {memory_reduction:.1f}%")

    # Anime per user in sampled data
    anime_per_user_sampled = sampled_df.groupby('user_id').size()

    print(f"\nAnime per User (Sampled):")
    print(f"  Mean: {anime_per_user_sampled.mean():.1f}")
    print(f"  Median: {anime_per_user_sampled.median():.1f}")
    print(f"  Min: {anime_per_user_sampled.min()}")
    print(f"  Max: {anime_per_user_sampled.max()}")
    print(f"  Std Dev: {anime_per_user_sampled.std():.1f}")

    print(f"\n✓ Data distribution preserved!")


def get_recommended_sample_size(original_info, target_memory_gb=2.0):
    """
    [HELPER] Calculate recommended sample size based on target memory

    Args:
    - original_info: Info dict from get_dataset_info()
    - target_memory_gb: Target memory size in GB

    Returns:
    - Recommended number of users to sample
    """
    total_users = original_info['unique_users']
    total_rows = original_info['total_rows']
    current_memory_gb = total_rows * 40 / 1e9  # Rough estimate

    # Calculate reduction ratio needed
    reduction_ratio = target_memory_gb / current_memory_gb

    # Scale down users accordingly
    recommended_users = int(total_users * reduction_ratio)

    # Estimate resulting rows and memory
    avg_anime_per_user = total_rows / total_users
    estimated_rows = int(recommended_users * avg_anime_per_user)
    estimated_memory = estimated_rows * 40 / 1e9

    return {
        'recommended_users': recommended_users,
        'estimated_rows': estimated_rows,
        'estimated_memory_gb': estimated_memory
    }


def save_sampled_dataset(sampled_df, output_file='user_animelist_sampled.csv'):
    """
    [STEP 4] Save the sampled dataset to CSV

    Args:
    - sampled_df: The sampled dataframe
    - output_file: Output filename
    """
    print("\n[STEP 4] Saving Sampled Dataset")
    print("-" * 70)

    print(f"\nSaving to {output_file}...")
    sampled_df.to_csv(output_file, index=False)
    print(f"✓ Sampled dataset saved!")
    print(f"  Rows: {len(sampled_df):,}")
    print(f"  File size: {sampled_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


# ============================================================================
# USAGE
# ============================================================================

def run_sampling(user_animelist_df, num_users=10000, output_file='user_animelist_sampled.csv'):
    """
    Run the complete sampling pipeline

    Args:
    - user_animelist_df: The full user animelist dataset
    - num_users: Number of users to sample
    - output_file: Where to save the sampled dataset

    Returns:
    - sampled_df: The sampled dataset
    """

    # Step 1: Analyze original dataset
    original_info = get_dataset_info(user_animelist_df)

    # Step 2: Get recommended sample size if needed
    print("\n[INFO] Recommended sample sizes for different memory targets:")
    print("-" * 70)
    for target_gb in [0.5, 1.0, 2.0, 3.0, 4.0]:
        rec = get_recommended_sample_size(original_info, target_gb)
        print(f"\nFor {target_gb} GB memory target:")
        print(f"  Sample ~{rec['recommended_users']:,} users")
        print(f"  Resulting in ~{rec['estimated_rows']:,} rows")
        print(f"  Estimated memory: {rec['estimated_memory_gb']:.2f} GB")

    # Step 3: Sample users
    sampled_df, sampled_user_ids = sample_users(user_animelist_df, num_users=num_users)

    # Step 4: Analyze sampled dataset
    analyze_sampled_dataset(sampled_df, original_info)

    # Step 5: Save sampled dataset
    save_sampled_dataset(sampled_df, output_file)

    print("\n" + "="*70)
    print("✓ SAMPLING COMPLETE!")
    print("="*70)
    print(f"\nNext steps:")
    print(f"1. Load the sampled dataset: user_animelist_sampled = pd.read_csv('{output_file}')")
    print(f"2. Proceed to Phase 3 with the sampled data")

    return sampled_df

In [ ]:
# See what sample size gives you what memory usage
original_info = get_dataset_info(user_completed_animelist)


USER ANIMELIST DATASET SAMPLING

[STEP 1] Current Dataset Information
----------------------------------------------------------------------

Total rows: 57,633,278
Unique users: 310,059
Unique anime: 16,872
Memory usage: 1.38 GB

Anime per User Statistics:
  Mean: 185.9
  Median: 113.0
  Min: 1
  Max: 15455
  Std Dev: 255.3


In [ ]:
# Then based on the recommendations, sample
sampled_df = run_sampling(user_completed_animelist, num_users=10000)  # Or however many you want


USER ANIMELIST DATASET SAMPLING

[STEP 1] Current Dataset Information
----------------------------------------------------------------------

Total rows: 57,633,278
Unique users: 310,059
Unique anime: 16,872
Memory usage: 1.38 GB

Anime per User Statistics:
  Mean: 185.9
  Median: 113.0
  Min: 1
  Max: 15455
  Std Dev: 255.3

[INFO] Recommended sample sizes for different memory targets:
----------------------------------------------------------------------

For 0.5 GB memory target:
  Sample ~67,248 users
  Resulting in ~12,499,952 rows
  Estimated memory: 0.50 GB

For 1.0 GB memory target:
  Sample ~134,496 users
  Resulting in ~24,999,904 rows
  Estimated memory: 1.00 GB

For 2.0 GB memory target:
  Sample ~268,993 users
  Resulting in ~49,999,994 rows
  Estimated memory: 2.00 GB

For 3.0 GB memory target:
  Sample ~403,489 users
  Resulting in ~74,999,899 rows
  Estimated memory: 3.00 GB

For 4.0 GB memory target:
  Sample ~537,986 users
  Resulting in ~99,999,989 rows
  Estimated 